# Plan your trip with Kayak

### Context
This project aims at creating an application which will recommend where travelers should plan their next holidays, based on weather in the upcoming week and hotel ratings. This application will focus on the top-35 cities to travel to in France according to One Week In.com.

### Methodology
The following steps of this project are:
- Data collection
    - Scrape hotel data from the booking.com website for the 35 destinations
    - Collect hotels GPS coordinates and 5-day weather forecasts through external APIs
- Clean, enrich and merge data:
    - Define a holiday climate index in order to rate the destinations (the holiday climate index is a weighted score based on temperature, cloud cover, precipitation, and wind conditions)
    - Create a single dataset combining both hotels and weather data
- Store the dataset in a datalake using AWS S3
- Create a data warehouse (MySQL database) and store the dataset inside in order to retrieve structured data to perform analysis
- Extract data retrieved from the SQL database to create maps displaying:
    - the top 5 cities where the weather will be the nicest within the next 5 days;
    - the top 20 best-rated hotels located in these destinations.


In [1]:
# Import packages

import os
import time
import boto3
from dotenv import load_dotenv
import json
import numpy as np
import pandas as pd
import plotly.colors as px_colors
import plotly.io as pio
import plotly.express as px
import requests
from sqlalchemy import create_engine

In [ ]:
# Load environment variables from the .env file

load_dotenv()

OPENWEATHERMAP_API_KEY = os.getenv("OPENWEATHERMAP_API_KEY")

AWS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_BUCKET_NAME = os.getenv("AWS_BUCKET_NAME")
DB_HOSTNAME = os.getenv("DB_HOSTNAME")
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

In [3]:
os.makedirs("data", exist_ok=True)

# 1. Data collection

## 1.1. Collecting hotel data

In this section, we scrape the Booking.com website for the top 35 destinations in France.

In [3]:
# List of destinations
city_list = [
"Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]

# Create a text file containing the destinations
textfile = open("data/city_list.txt", "w")
for element in city_list:
    textfile.write(element + "\n")
textfile.close()


In [4]:
!python scrape_hotel.py


--- City: Mont Saint Michel ---
Cookies accepted
25 hotels found
La Vieille Auberge
https://www.booking.com/hotel/fr/la-vieille-auberge-le-mont-saint-michel.fr.html?aid=304142&label=mkt123sc-5f611bb5-07cf-43e4-bbcc-a3de372e848d&ucfs=1&arphpl=1&group_adults=2&req_adults=2&no_rooms=1&group_children=0&req_children=0&hpos=1&hapos=1&sr_order=popularity&srpvid=aa1657369490a310a09acd3c2126c42a&srepoch=1766846970&from=searchresults
Rating: 7.5
Reviews: 1603
Description: La Vieille Auberge vous accueille dans le village médiéval du Mont-Saint-Michel, à quelques pas de la célèbre abbaye. Vous pourrez profiter d’une connexion Wi-Fi gratuite, d’un restaurant, d’une terrasse et d’une réception dans la maison principale.

Les chambres sont réparties dans 2 annexes situées en haut du village. Pour y accéder, vous devrez gravir de nombreuses marches. L’une des options, au milieu du village, propose des chambres standard donnant sur la rue, tandis que l’autre option, en haut du village, propose des ch

In [9]:
# Create city_id
df_city = pd.DataFrame({
    "city_id": range(1, len(city_list) + 1),
    "city": city_list
})

df_city.head()

,city_id,city
0,1,Mont Saint Michel
1,2,St Malo
2,3,Bayeux
3,4,Le Havre
4,5,Rouen


In [10]:
# Load file with hotels information from Booking.com 

with open("data/hotel_info.json", "r", encoding="utf-8") as file_hotel:
    data = json.load(file_hotel)

df_hotel = pd.DataFrame(data)

df_hotel.head()

,city,hotel_name,hotel_url,hotel_latitude,hotel_longitude,hotel_rating,hotel_reviews,hotel_description
0,Mont Saint Michel,La Vieille Auberge,https://www.booking.com/hotel/fr/la-vieille-au...,48.636063,-1.511457,7.5,1603.0,La Vieille Auberge vous accueille dans le vill...
1,Mont Saint Michel,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,48.614247,-1.510545,8.3,3662.0,Installé dans des espaces verts à seulement 2 ...
2,Mont Saint Michel,Auberge Saint Pierre,https://www.booking.com/hotel/fr/auberge-saint...,48.635688,-1.509883,8.1,1215.0,"Située sur le Mont-Saint-Michel, l'Auberge Sai..."
3,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html?...,48.614700,-1.509617,8.2,6099.0,L’Hotel Vert vous propose des chambres décorée...
4,Mont Saint Michel,Appart Standing - La Coque d'Or - Mont-St-Michel,https://www.booking.com/hotel/fr/la-coque-d-or...,48.635487,-1.510155,9.6,56.0,L’hébergement Appart Standing - La Coque d'Or ...


In [11]:
# Add city_id

print(f"Before merge: {len(df_hotel)} rows")

df_hotel = df_hotel.merge(df_city,
                        how='left', 
                        on=['city'])

print(f"After merge:{len(df_hotel)} rows")

Before merge: 175 rows
After merge:175 rows


## 1.2. Collecting GPS coordinates and weather data using APIs

### 1.2.1. GPS coordinates

In [12]:
# Use free-form query because not all destinations are cities

headers = {
    'User-Agent': 'My User Agent',
}

rows = []

for city_id, city in zip(df_city["city_id"], df_city["city"]):

    params = {
        "q": f"{city}, France",
        "format": "jsonv2"
    }

    if city == "Mont Saint Michel":
        params["q"] = f"{city}, Normandie, France"

    r = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params=params,
        headers=headers
    )

    if r.status_code != 200:
        print(f"Error for {city} ({r.status_code})")
        continue

    data = r.json()
    
    if not data:
        print(f"No result for {city}")
        continue

    df_temp = pd.DataFrame(data)
    df_temp["city"] = city
    df_temp["city_id"] = city_id

    rows.append(df_temp)

    print(f"{city}: query done")

    time.sleep(1)


df_gps = pd.concat(rows, ignore_index=True)

Mont Saint Michel: query done
St Malo: query done
Bayeux: query done
Le Havre: query done
Rouen: query done
Paris: query done
Amiens: query done
Lille: query done
Strasbourg: query done
Chateau du Haut Koenigsbourg: query done
Colmar: query done
Eguisheim: query done
Besancon: query done
Dijon: query done
Annecy: query done
Grenoble: query done
Lyon: query done
Gorges du Verdon: query done
Bormes les Mimosas: query done
Cassis: query done
Marseille: query done
Aix en Provence: query done
Avignon: query done
Uzes: query done
Nimes: query done
Aigues Mortes: query done
Saintes Maries de la mer: query done
Collioure: query done
Carcassonne: query done
Ariege: query done
Toulouse: query done
Montauban: query done
Biarritz: query done
Bayonne: query done
La Rochelle: query done


In [13]:
# Rename coordinates in order to not confuse with hotel coordinates (useful further in the merging process)
df_gps.rename(columns = {'lat' : 'city_lat', 
                        'lon' : 'city_lon',
                        'display_name' : 'city_display_name'},
            inplace = True)

In [14]:
# Display shape of coordinates dataframe
print(df_gps.shape)

(63, 16)


In [15]:
df_gps.head(5)

,place_id,licence,osm_type,osm_id,city_lat,city_lon,category,type,place_rank,importance,addresstype,name,city_display_name,boundingbox,city,city_id
0,262508722,"Data © OpenStreetMap contributors, ODbL 1.0. h...",way,211285890,48.6359541,-1.5114600,place,islet,20,0.472371,islet,Mont Saint-Michel,"Mont Saint-Michel, Le Mont-Saint-Michel, Avran...","[48.6349172, 48.6370310, -1.5133292, -1.5094796]",Mont Saint Michel,1
1,262113556,"Data © OpenStreetMap contributors, ODbL 1.0. h...",node,5972710523,48.6360211,-1.5114947,natural,peak,18,0.160035,peak,Mont Saint-Michel,"Mont Saint-Michel, Le Mont-Saint-Michel, Avran...","[48.6359711, 48.6360711, -1.5115447, -1.5114447]",Mont Saint Michel,1
2,262173744,"Data © OpenStreetMap contributors, ODbL 1.0. h...",relation,905534,48.6495180,-2.0260409,boundary,administrative,16,0.625099,town,Saint-Malo,"Saint-Malo, Ille-et-Vilaine, Bretagne, France ...","[48.5979853, 48.6949736, -2.0765246, -1.9367259]",St Malo,2
3,261847659,"Data © OpenStreetMap contributors, ODbL 1.0. h...",relation,1653637,48.4904728,-1.7421555,boundary,administrative,14,0.506196,municipality,Saint-Malo,"Saint-Malo, Ille-et-Vilaine, Bretagne, France ...","[48.2669570, 48.7220079, -2.1619838, -1.4850797]",St Malo,2
4,261770605,"Data © OpenStreetMap contributors, ODbL 1.0. h...",relation,145776,49.2764624,-0.7024738,boundary,administrative,16,0.604295,town,Bayeux,"Bayeux, Calvados, Normandie, France métropolit...","[49.2608110, 49.2934736, -0.7275523, -0.6757378]",Bayeux,3


In [16]:
# Drop duplicates, because each query can give several results. 
# Select the first result, which is associated with the lowest place_rank
df_gps = df_gps.sort_values(['city_id', 'place_rank']).drop_duplicates(['city_id']).reset_index(drop=True)

df_gps = df_gps[['city_id', 'city', 'city_display_name', 'city_lat', 'city_lon']]

df_gps.head()

,city_id,city,city_display_name,city_lat,city_lon
0,1,Mont Saint Michel,"Mont Saint-Michel, Le Mont-Saint-Michel, Avran...",48.6360211,-1.5114947
1,2,St Malo,"Saint-Malo, Ille-et-Vilaine, Bretagne, France ...",48.4904728,-1.7421555
2,3,Bayeux,"Bayeux, Calvados, Normandie, France métropolit...",49.2455634,-0.7916770
3,4,Le Havre,"Le Havre, Seine-Maritime, Normandie, France mé...",49.6275492,0.3921241
4,5,Rouen,"Rouen, Seine-Maritime, Normandie, France métro...",49.5184350,1.0221643


In [ ]:
df_gps.to_csv("data/gps.csv", index = False)

### 1.2.2. Weather data

We use free access data from the [5 day weather forecast API](https://openweathermap.org/forecast5), which displays weather forecast for 5 days with 3-hour step.

We fetch weather data for the 5 next days from the API, in a JSON format.


In [19]:
# lists of cities, latitudes, longitudes
city_id_list = df_gps['city_id'].tolist()
city_list = df_gps['city'].tolist()
lat_list = df_gps['city_lat'].tolist()
lon_list = df_gps['city_lon'].tolist()

In [20]:
# Initialize a list to store the results of the API requests
list_results = []


# Loop which fetches weather data for each destination

for city_id, city, lat, lon in zip(city_id_list, city_list, lat_list, lon_list):
    
    print(f"Fetch weather data for {city}")
    
    params = {
        "lat": lat,
        "lon": lon,
        "units": "metric",
        "lang": "fr",
        "appid": OPENWEATHERMAP_API_KEY
    }
    
    r = requests.get(
        "https://api.openweathermap.org/data/2.5/forecast",
        params=params
    )
    
    if r.status_code != 200:
        print(f"Issue encountered for {city}: {r.status_code}")
        continue

    json_data = r.json()
    forecasts = json_data['list']

    if not forecasts:
        print(f"No forecast data for {city}")
        continue
    
    df = pd.json_normalize(
        forecasts,
        record_path=["weather"],
        meta=[
            "dt", "dt_txt", "visibility", "pop",
            ["main", "temp"], ["main", "feels_like"],
            ["main", "temp_min"], ["main", "temp_max"],
            ["main", "pressure"], ["main", "humidity"],
            ["wind", "speed"], ["wind", "deg"], ["wind", "gust"],
            ["rain", "3h"],
            ["clouds", "all"], ["sys", "pod"]
        ],
        record_prefix="weather_",
        sep="_",
        errors="ignore"
    )

    df = df.assign(
        city_id=city_id,
        city=city,
        city_lat=lat,
        city_lon=lon,
        dt=pd.to_datetime(df["dt"], unit="s")
    )

    list_results.append(df)
    time.sleep(1)

if list_results:
    df_weather = pd.concat(list_results, ignore_index=True)
    df_weather.to_csv("data/weather.csv", index=False)
else:
    print("No weather data collected.")

Fetch weather data for Mont Saint Michel
Fetch weather data for St Malo
Fetch weather data for Bayeux
Fetch weather data for Le Havre
Fetch weather data for Rouen
Fetch weather data for Paris
Fetch weather data for Amiens
Fetch weather data for Lille
Fetch weather data for Strasbourg
Fetch weather data for Chateau du Haut Koenigsbourg
Fetch weather data for Colmar
Fetch weather data for Eguisheim
Fetch weather data for Besancon
Fetch weather data for Dijon
Fetch weather data for Annecy
Fetch weather data for Grenoble
Fetch weather data for Lyon
Fetch weather data for Gorges du Verdon
Fetch weather data for Bormes les Mimosas
Fetch weather data for Cassis
Fetch weather data for Marseille
Fetch weather data for Aix en Provence
Fetch weather data for Avignon
Fetch weather data for Uzes
Fetch weather data for Nimes
Fetch weather data for Aigues Mortes
Fetch weather data for Saintes Maries de la mer
Fetch weather data for Collioure
Fetch weather data for Carcassonne
Fetch weather data for A

In [21]:
df_weather.head()

,weather_id,weather_main,weather_description,weather_icon,dt,dt_txt,visibility,pop,main_temp,main_feels_like,...,wind_speed,wind_deg,wind_gust,rain_3h,clouds_all,sys_pod,city_id,city,city_lat,city_lon
0,800,Clear,ciel dégagé,01n,2025-12-28 00:00:00,2025-12-28 00:00:00,10000,0,1.12,-2.17,...,3.02,76,5,NaN,1,n,1,Mont Saint Michel,48.6360211,-1.5114947
1,800,Clear,ciel dégagé,01n,2025-12-28 03:00:00,2025-12-28 03:00:00,10000,0,1.27,-2.09,...,3.14,62,4.52,NaN,0,n,1,Mont Saint Michel,48.6360211,-1.5114947
2,800,Clear,ciel dégagé,01n,2025-12-28 06:00:00,2025-12-28 06:00:00,10000,0,1.75,-1.74,...,3.44,55,5.37,NaN,1,n,1,Mont Saint Michel,48.6360211,-1.5114947
3,800,Clear,ciel dégagé,01d,2025-12-28 09:00:00,2025-12-28 09:00:00,10000,0,2.96,-0.51,...,3.78,58,9.8,NaN,0,d,1,Mont Saint Michel,48.6360211,-1.5114947
4,800,Clear,ciel dégagé,01d,2025-12-28 12:00:00,2025-12-28 12:00:00,10000,0,6.62,3.17,...,5.44,64,8.06,NaN,0,d,1,Mont Saint Michel,48.6360211,-1.5114947


In [22]:
df_weather.dtypes

weather_id                      int64
weather_main                   object
weather_description            object
weather_icon                   object
dt                     datetime64[ns]
dt_txt                         object
visibility                     object
pop                            object
main_temp                      object
main_feels_like                object
main_temp_min                  object
main_temp_max                  object
main_pressure                  object
main_humidity                  object
wind_speed                     object
wind_deg                       object
wind_gust                      object
rain_3h                        object
clouds_all                     object
sys_pod                        object
city_id                         int64
city                           object
city_lat                       object
city_lon                       object
dtype: object

# 2. Cleaning and enrichment phase

## 2.1. Grouping the 3 hour forecasts into daily forecasts
For each day, compute the average for the following indicators: estimated sunshine hours, temperature, humidity, rain, wind speed.

In [23]:
df_weather = pd.read_csv("data/weather.csv")

print(f"Shape of weather dataframe : {df_weather.shape}\n")
print(f"Number of missing values per variable:\n{df_weather.isnull().sum()}\n")
print(f"Variables types:\n{df_weather.dtypes}")

Shape of weather dataframe : (1400, 24)

Number of missing values per variable:
weather_id                0
weather_main              0
weather_description       0
weather_icon              0
dt                        0
dt_txt                    0
visibility                1
pop                       0
main_temp                 0
main_feels_like           0
main_temp_min             0
main_temp_max             0
main_pressure             0
main_humidity             0
wind_speed                0
wind_deg                  0
wind_gust                 0
rain_3h                1368
clouds_all                0
sys_pod                   0
city_id                   0
city                      0
city_lat                  0
city_lon                  0
dtype: int64

Variables types:
weather_id               int64
weather_main            object
weather_description     object
weather_icon            object
dt                      object
dt_txt                  object
visibility             float64


In [24]:
df_weather['dt'] = pd.to_datetime(df_weather['dt'])
df_weather['date'] =  df_weather['dt'].dt.date

df_weather['rain_3h'] = df_weather['rain_3h'].fillna(0)

In [25]:
# Create dataframe which aggregates 3-hours forecasts into daily forecasts
cols_to_keep = [
    'city_id', 'city', 'city_lat', 'city_lon', 'date', 
    'main_temp', 'main_feels_like',
    'wind_speed', 'rain_3h', 'clouds_all'
    ]

cols_to_group = ['city_id', 'city', 'city_lat', 'city_lon', 'date']

df_weather_daily = df_weather[cols_to_keep].groupby(cols_to_group, dropna=False).mean(numeric_only=True).reset_index()

print("Shape of daily weather dataframe:", df_weather_daily.shape)
df_weather_daily.head()

Shape of daily weather dataframe: (175, 10)


,city_id,city,city_lat,city_lon,date,main_temp,main_feels_like,wind_speed,rain_3h,clouds_all
0,1,Mont Saint Michel,48.636021,-1.511495,2025-12-28,2.85875,-0.59750,3.83625,0.0,3.250
1,1,Mont Saint Michel,48.636021,-1.511495,2025-12-29,1.38875,-2.01625,3.27500,0.0,3.375
2,1,Mont Saint Michel,48.636021,-1.511495,2025-12-30,3.76375,-0.42250,5.69375,0.0,33.875
3,1,Mont Saint Michel,48.636021,-1.511495,2025-12-31,1.09625,-2.49375,3.41750,0.0,6.375
4,1,Mont Saint Michel,48.636021,-1.511495,2026-01-01,0.07875,-1.99250,1.97500,0.0,76.500


## 2.2. Defining a Holiday Climate Index (HCI)

The **Holiday Climate Index (HCI)** has been defined by Scott, D., Rutty, M., Amelung, B., Tang, M. in their article "An inter-comparison of the Holiday Climate Index (HCI) and the Tourism Climate Index (TCI) in Europe" (*Atmosphere* 2016, [link](http://doi.org/10.3390/atmos7060080)). 

In this research paper, the authors define an index for urban tourism, based on stated tourist climate preferences obtained from surveys. 

The HCI index encompasses different aspects of climate important to leisure tourism activities: 
- **thermal comfort** ($TC$), which corresponds to effective temperature; 
- **aesthetic** ($A$) which corresponds to cloud cover (%); 
- **physical component**, which is a combination of precipitation $P$ (mm) and wind speed $W$ (km/h). 

The HCI is calculated as :

$$ HCI = 4 TC + 2A + 3 P + W $$

The HCI components (TC, A, P, W) are not computed directly from raw meteorological values.
Observed weather variables are first mapped to **ordinal rating scales**, based on predefined thresholds derived from the HCI methodology. These ratings are intended to reflect tourists perceived comfort.

The HCI varies from -11 (dangerous for tourism) to 100 (ideal). Below a score of 40, the conditions of tourism are defined as inacceptable for the majority of tourists.

| HCI score  | Description          | 
| :--------------- |:---------------:| 
| 90 - 100  |   ideal        |  
| 80 - 89  | excellent             | 
| 70 - 79  | very good       |    
| 60 - 69  | good          |    
| 40 - 59  | marginal          |       
| 20 - 39  | unacceptable          |    
| -11 - 19  | dangerous          |     

In [26]:
# Mapping values: convert felt-like temperature, cloud cover, precipitation and wind speed into their rating scale

def condition_temp(temp):
    if temp >= 39:
        return 0
    elif temp >= 37:
        return 2
    elif temp >= 35:
        return 4
    elif temp >= 33:
        return 5
    elif temp >= 31:
        return 6
    elif temp >= 29:
        return 7
    elif temp >= 27:
        return 8
    elif temp == 26:
        return 9
    elif temp >= 23:
        return 10
    elif temp >= 20:
        return 9
    elif temp >= 18:
        return 7
    elif temp >= 15:
        return 6
    elif temp >= 11:
        return 5
    elif temp >= 7:
        return 4
    elif temp >= 0:
        return 3
    elif temp > -6:
        return 2
    else:
        return 1

# Vectorize the function
func_temp = np.vectorize(condition_temp)


def condition_cloud(cloud):
    if cloud >= 100:
        return 1
    elif cloud >= 90:
        return 2
    elif cloud >= 81:
        return 3
    elif cloud >= 71:
        return 4
    elif cloud >= 61:
        return 5
    elif cloud >= 51:
        return 6
    elif cloud >= 41:
        return 7
    elif cloud >= 31:
        return 8
    elif cloud >= 21:
        return 9
    elif cloud >= 11:
        return 10
    elif cloud >= 1:
        return 9
    elif cloud >= 0:
        return 8
    else:
        return 999

# Vectorize the function
func_cloud = np.vectorize(condition_cloud)


def condition_rain(rain):
    if rain >= 25:
        return -1
    elif rain >= 12:
        return 0
    elif rain >= 9:
        return 2
    elif rain >= 6:
        return 5
    elif rain >= 3:
        return 8
    elif rain >= 0.01:
        return 9
    elif rain >= 0:
        return 10
    else:
        return 999
    
# Vectorize the function
func_rain = np.vectorize(condition_rain)


def condition_wind(wind):
    if wind >= 70:
        return -10
    elif wind >= 50:
        return 0
    elif wind >= 40:
        return 3
    elif wind >= 30:
        return 6
    elif wind >= 20:
        return 8
    elif wind >= 10:
        return 9
    elif wind >= 0.02:
        return 10
    elif wind >= 0:
        return 8
    else:
        return 999 

# Vectorize the function
func_wind = np.vectorize(condition_wind)


def condition_hci(score):
    if score >= 90:
        return "ideal"
    elif score >= 80:
        return "excellent"
    elif score >= 70:
        return "very good"
    elif score >= 60:
        return "good"
    elif score >= 50:
        return "acceptable"
    elif score >= 40:
        return "marginal"
    elif score >= 20:
        return "unacceptable"
    elif score >= -11:
        return "dangerous"
    else:
        return "undefined"

# Vectorize the function
func_hci = np.vectorize(condition_hci)

In [27]:
# Create climatic variables and HCI score
df_weather_daily["thermal_comfort_score"] = func_temp(df_weather_daily["main_feels_like"])
df_weather_daily["cloud_score"] = func_cloud(df_weather_daily['clouds_all'])
df_weather_daily["rain_score"] = func_rain(df_weather_daily['rain_3h'])
df_weather_daily["wind_score"] = func_wind(df_weather_daily['wind_speed'])
df_weather_daily['hci'] =  4 * df_weather_daily['thermal_comfort_score'] + 2 * df_weather_daily['cloud_score']\
    + 3 * df_weather_daily['rain_score'] + df_weather_daily['wind_score']


In [28]:
df_weather_daily.head()

,city_id,city,city_lat,city_lon,date,main_temp,main_feels_like,wind_speed,rain_3h,clouds_all,thermal_comfort_score,cloud_score,rain_score,wind_score,hci
0,1,Mont Saint Michel,48.636021,-1.511495,2025-12-28,2.85875,-0.59750,3.83625,0.0,3.250,2,9,10,10,66
1,1,Mont Saint Michel,48.636021,-1.511495,2025-12-29,1.38875,-2.01625,3.27500,0.0,3.375,2,9,10,10,66
2,1,Mont Saint Michel,48.636021,-1.511495,2025-12-30,3.76375,-0.42250,5.69375,0.0,33.875,2,8,10,10,64
3,1,Mont Saint Michel,48.636021,-1.511495,2025-12-31,1.09625,-2.49375,3.41750,0.0,6.375,2,9,10,10,66
4,1,Mont Saint Michel,48.636021,-1.511495,2026-01-01,0.07875,-1.99250,1.97500,0.0,76.500,2,4,10,10,56


In [29]:
# Defining the average HCI score over the 5 next days for each city
df_weather_avg = df_weather_daily\
    .groupby(['city_id', 'city', 'city_lat', 'city_lon'], dropna=False)[
        ['hci', 'main_temp', 'main_feels_like', 'wind_speed', 'rain_3h', 'clouds_all']
        ]\
    .mean()\
    .reset_index()

df_weather_avg.head()

,city_id,city,city_lat,city_lon,hci,main_temp,main_feels_like,wind_speed,rain_3h,clouds_all
0,1,Mont Saint Michel,48.636021,-1.511495,63.6,1.83725,-1.50450,3.63950,0.0,24.675
1,2,St Malo,48.490473,-1.742155,63.6,1.19300,-2.25400,3.49025,0.0,24.975
2,3,Bayeux,49.245563,-0.791677,62.4,1.98275,-1.67350,4.15775,0.0,44.175
3,4,Le Havre,49.627549,0.392124,63.2,1.60975,-1.94050,3.93175,0.0,36.075
4,5,Rouen,49.518435,1.022164,63.2,0.89500,-2.34325,3.44425,0.0,34.100


In [30]:
df_weather_avg = df_weather_avg.rename(columns={
    'main_temp':'temp',
    'main_feels_like':'temp_feels_like',
    'rain_3h':'rain',
    'clouds_all':'cloud'
})

In [31]:
df_weather_avg.head()

,city_id,city,city_lat,city_lon,hci,temp,temp_feels_like,wind_speed,rain,cloud
0,1,Mont Saint Michel,48.636021,-1.511495,63.6,1.83725,-1.50450,3.63950,0.0,24.675
1,2,St Malo,48.490473,-1.742155,63.6,1.19300,-2.25400,3.49025,0.0,24.975
2,3,Bayeux,49.245563,-0.791677,62.4,1.98275,-1.67350,4.15775,0.0,44.175
3,4,Le Havre,49.627549,0.392124,63.2,1.60975,-1.94050,3.93175,0.0,36.075
4,5,Rouen,49.518435,1.022164,63.2,0.89500,-2.34325,3.44425,0.0,34.100


## 2.3. Creating the final dataset: merging hotel and weather data

We merge all the information (hotels' characteristics and weather forecasts) into a unique DataFrame, and store it in a CSV file.

In [32]:
print(f"Before merge,\n shape of left df: {df_hotel.shape}\n")
print(f"shape of left df: {df_weather_avg.shape}\n")

# The merge variable is the city ID
df_final = df_hotel.merge(df_weather_avg.drop(columns=['city']),
                        how='left', 
                        on=['city_id'])

print(f"Shape of final dataset: {df_final.shape}")

df_final.head()

Before merge,
 shape of left df: (175, 9)

shape of left df: (35, 10)

Shape of final dataset: (175, 17)


,city,hotel_name,hotel_url,hotel_latitude,hotel_longitude,hotel_rating,hotel_reviews,hotel_description,city_id,city_lat,city_lon,hci,temp,temp_feels_like,wind_speed,rain,cloud
0,Mont Saint Michel,La Vieille Auberge,https://www.booking.com/hotel/fr/la-vieille-au...,48.636063,-1.511457,7.5,1603.0,La Vieille Auberge vous accueille dans le vill...,1,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
1,Mont Saint Michel,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,48.614247,-1.510545,8.3,3662.0,Installé dans des espaces verts à seulement 2 ...,1,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
2,Mont Saint Michel,Auberge Saint Pierre,https://www.booking.com/hotel/fr/auberge-saint...,48.635688,-1.509883,8.1,1215.0,"Située sur le Mont-Saint-Michel, l'Auberge Sai...",1,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
3,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html?...,48.614700,-1.509617,8.2,6099.0,L’Hotel Vert vous propose des chambres décorée...,1,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
4,Mont Saint Michel,Appart Standing - La Coque d'Or - Mont-St-Michel,https://www.booking.com/hotel/fr/la-coque-d-or...,48.635487,-1.510155,9.6,56.0,L’hébergement Appart Standing - La Coque d'Or ...,1,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675


In [33]:
print(f"Variables types : {df_final.dtypes}")

Variables types : city                  object
hotel_name            object
hotel_url             object
hotel_latitude       float64
hotel_longitude      float64
hotel_rating         float64
hotel_reviews        float64
hotel_description     object
city_id                int64
city_lat             float64
city_lon             float64
hci                  float64
temp                 float64
temp_feels_like      float64
wind_speed           float64
rain                 float64
cloud                float64
dtype: object


In [37]:
# Extract the city_id column 
city_id_col = df_final.pop('city_id')

# Re-insert city_id at the beginning
df_final.insert(0, 'city_id', city_id_col)

In [38]:
df_final.head()

,city_id,city,hotel_name,hotel_url,hotel_latitude,hotel_longitude,hotel_rating,hotel_reviews,hotel_description,city_lat,city_lon,hci,temp,temp_feels_like,wind_speed,rain,cloud
0,1,Mont Saint Michel,La Vieille Auberge,https://www.booking.com/hotel/fr/la-vieille-au...,48.636063,-1.511457,7.5,1603.0,La Vieille Auberge vous accueille dans le vill...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
1,1,Mont Saint Michel,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,48.614247,-1.510545,8.3,3662.0,Installé dans des espaces verts à seulement 2 ...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
2,1,Mont Saint Michel,Auberge Saint Pierre,https://www.booking.com/hotel/fr/auberge-saint...,48.635688,-1.509883,8.1,1215.0,"Située sur le Mont-Saint-Michel, l'Auberge Sai...",48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
3,1,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html?...,48.614700,-1.509617,8.2,6099.0,L’Hotel Vert vous propose des chambres décorée...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
4,1,Mont Saint Michel,Appart Standing - La Coque d'Or - Mont-St-Michel,https://www.booking.com/hotel/fr/la-coque-d-or...,48.635487,-1.510155,9.6,56.0,L’hébergement Appart Standing - La Coque d'Or ...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675


In [39]:
df_final.sort_values('city_id', ignore_index=True, inplace=True)
df_final.to_csv("data/hotel_weather_final.csv", index=False)

# 3. Storing the database in a datalake (AWS S3 bucket)
The S3 bucket `kayak-project-formation` has been created beforehand.
In the following code, we store the final database into it using `boto3`.

In [43]:
# Set up a session
session = boto3.Session(aws_access_key_id = AWS_KEY_ID, 
                        aws_secret_access_key = AWS_SECRET)

# Set up a resource
s3 = session.resource("s3")

# Reference to the existing bucket
bucket = s3.Bucket(AWS_BUCKET_NAME) 

# Upload the final dataset to the bucket which was created beforehand
bucket.upload_file(
    Filename='data/hotel_weather_final.csv',
    Key='hotel_weather_final.csv'
)

# 4. Creation of a relational database
We send the data from AWS S3 bucket to AWS RDS in order to transform the dataset into a relational database which can be efficiently queried by the analytics team using SQL. This approch ensures a clear separation between the datalake (dedicated to data storage) and the data warehouse (optimized for analytical queries).

In this project we choose to create a MySQL database.

First, we use AWS RDS to create a SQL database instance and configure security rules.
Then we use SQL Alchemy to create an engine which will create a connection between the database and Python.
Finally, we push the dataframe to the SQL database using Pandas `to_sql` method.

In [47]:
# Create an sqlalchemy engine which will create a connection to a MySQL database and Python

engine = create_engine(
    f"mysql+pymysql://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOSTNAME}:3306/{DB_NAME}",
    echo=True
)

In [48]:
# Test connection
with engine.connect() as conn:
    print("Connection successful")

2025-12-27 23:55:40,587 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2025-12-27 23:55:40,590 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-12-27 23:55:40,630 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2025-12-27 23:55:40,632 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-12-27 23:55:40,653 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2025-12-27 23:55:40,655 INFO sqlalchemy.engine.Engine [raw sql] {}
Connection successful


In [49]:
# Extract the dataset from the S3 bucket

dataset_key = "hotel_weather_final.csv"

df_s3 = pd.read_csv(
    f"s3://{AWS_BUCKET_NAME}/{dataset_key}",
    storage_options={
        "key": os.getenv("AWS_KEY_ID"),
        "secret": os.getenv("AWS_SECRET")
    }
)

df_s3.head()

,city_id,city,hotel_name,hotel_url,hotel_latitude,hotel_longitude,hotel_rating,hotel_reviews,hotel_description,city_lat,city_lon,hci,temp,temp_feels_like,wind_speed,rain,cloud
0,1,Mont Saint Michel,La Vieille Auberge,https://www.booking.com/hotel/fr/la-vieille-au...,48.636063,-1.511457,7.5,1603.0,La Vieille Auberge vous accueille dans le vill...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
1,1,Mont Saint Michel,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,48.614247,-1.510545,8.3,3662.0,Installé dans des espaces verts à seulement 2 ...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
2,1,Mont Saint Michel,Auberge Saint Pierre,https://www.booking.com/hotel/fr/auberge-saint...,48.635688,-1.509883,8.1,1215.0,"Située sur le Mont-Saint-Michel, l'Auberge Sai...",48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
3,1,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html?...,48.614700,-1.509617,8.2,6099.0,L’Hotel Vert vous propose des chambres décorée...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
4,1,Mont Saint Michel,Appart Standing - La Coque d'Or - Mont-St-Michel,https://www.booking.com/hotel/fr/la-coque-d-or...,48.635487,-1.510155,9.6,56.0,L’hébergement Appart Standing - La Coque d'Or ...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675


In [50]:
# Store the dataset into the SQL database
TABLE_NAME = "hotel_weather_final"
df_s3.to_sql(
    TABLE_NAME,
    con=engine,
    if_exists='replace',
    index=False
)

print(f"File uploaded in {TABLE_NAME} of the database {DB_NAME}")

2025-12-28 00:18:21,016 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-12-28 00:18:21,023 INFO sqlalchemy.engine.Engine DESCRIBE `kayak_mysql_db`.`hotel_weather_final`
2025-12-28 00:18:21,024 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-12-28 00:18:21,073 INFO sqlalchemy.engine.Engine 
CREATE TABLE hotel_weather_final (
	city_id BIGINT, 
	city TEXT, 
	hotel_name TEXT, 
	hotel_url TEXT, 
	hotel_latitude FLOAT(53), 
	hotel_longitude FLOAT(53), 
	hotel_rating FLOAT(53), 
	hotel_reviews FLOAT(53), 
	hotel_description TEXT, 
	city_lat FLOAT(53), 
	city_lon FLOAT(53), 
	hci FLOAT(53), 
	temp FLOAT(53), 
	temp_feels_like FLOAT(53), 
	wind_speed FLOAT(53), 
	rain FLOAT(53), 
	cloud FLOAT(53)
)


2025-12-28 00:18:21,075 INFO sqlalchemy.engine.Engine [no key 0.00227s] {}
2025-12-28 00:18:21,178 INFO sqlalchemy.engine.Engine INSERT INTO hotel_weather_final (city_id, city, hotel_name, hotel_url, hotel_latitude, hotel_longitude, hotel_rating, hotel_reviews, hotel_description, city_lat, city

# 5. Data visualisation

We retrieve data related to the top 5 destinations from the PostgreSQL database, and create two maps which display:
- the top 5 destinations
- the top 20 best-rated hotels located in these destinations

In [51]:
# Retrieve dataset from MySQL database
df_mysql = pd.read_sql("SELECT * FROM hotel_weather_final", con=engine)
df_mysql.head()

2025-12-28 00:33:32,275 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-12-28 00:33:32,277 INFO sqlalchemy.engine.Engine DESCRIBE `kayak_mysql_db`.`SELECT * FROM hotel_weather_final`
2025-12-28 00:33:32,279 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-12-28 00:33:32,303 INFO sqlalchemy.engine.Engine SELECT * FROM hotel_weather_final
2025-12-28 00:33:32,305 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-12-28 00:33:32,560 INFO sqlalchemy.engine.Engine ROLLBACK


,city_id,city,hotel_name,hotel_url,hotel_latitude,hotel_longitude,hotel_rating,hotel_reviews,hotel_description,city_lat,city_lon,hci,temp,temp_feels_like,wind_speed,rain,cloud
0,1,Mont Saint Michel,La Vieille Auberge,https://www.booking.com/hotel/fr/la-vieille-au...,48.636063,-1.511457,7.5,1603.0,La Vieille Auberge vous accueille dans le vill...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
1,1,Mont Saint Michel,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,48.614247,-1.510545,8.3,3662.0,Installé dans des espaces verts à seulement 2 ...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
2,1,Mont Saint Michel,Auberge Saint Pierre,https://www.booking.com/hotel/fr/auberge-saint...,48.635688,-1.509883,8.1,1215.0,"Située sur le Mont-Saint-Michel, l'Auberge Sai...",48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
3,1,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html?...,48.614700,-1.509617,8.2,6099.0,L’Hotel Vert vous propose des chambres décorée...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675
4,1,Mont Saint Michel,Appart Standing - La Coque d'Or - Mont-St-Michel,https://www.booking.com/hotel/fr/la-coque-d-or...,48.635487,-1.510155,9.6,56.0,L’hébergement Appart Standing - La Coque d'Or ...,48.636021,-1.511495,63.6,1.83725,-1.5045,3.6395,0.0,24.675


In [52]:
# Top 5 destinations according to the Holiday Climate Index
df_top5_city = df_mysql[['city', 'city_id', 'city_lat', 'city_lon', 'hci', 'temp', 'wind_speed', 'rain', 'cloud']].drop_duplicates(['city'])\
    .sort_values('hci', ascending=False, ignore_index=True).iloc[:5]
df_top5_city

,city,city_id,city_lat,city_lon,hci,temp,wind_speed,rain,cloud
0,Cassis,20,43.214036,5.539632,69.4,9.34600,3.34050,0.04200,23.225
1,Aigues Mortes,26,43.566152,4.191540,68.4,6.65950,4.32275,0.00000,12.550
2,Montauban,32,44.054015,1.493887,68.0,5.03100,2.07700,0.00000,31.225
3,Marseille,21,43.301488,5.548001,67.8,8.73925,2.64375,0.03525,22.450
4,Saintes Maries de la mer,27,43.451592,4.427720,67.6,7.22325,6.71400,0.00300,18.775


In [54]:
# Top 20 best-rated hotels located in these destinations
df_top20_hotel = df_mysql.loc[df_final['city_id'].isin(df_top5_city['city_id'])]\
    .sort_values(['hotel_rating', 'hotel_reviews'], ascending=False, ignore_index=True).iloc[:20]
df_top20_hotel

,city_id,city,hotel_name,hotel_url,hotel_latitude,hotel_longitude,hotel_rating,hotel_reviews,hotel_description,city_lat,city_lon,hci,temp,temp_feels_like,wind_speed,rain,cloud
0,27,Saintes Maries de la mer,Résidence L'Oiseau des Mers,https://www.booking.com/hotel/fr/residence-l-o...,43.454050,4.427996,9.9,73.0,L’établissement Résidence L'Oiseau des Mers se...,43.451592,4.427720,67.6,7.22325,3.6560,6.71400,0.00300,18.775
1,20,Cassis,Le Bel Ecrin par Dodo-a-Cassis,https://www.booking.com/hotel/fr/le-bon-crin.f...,43.214187,5.534756,9.7,47.0,"Situé à Cassis, l’hébergement Le Bel Ecrin par...",43.214036,5.539632,69.4,9.34600,7.4960,3.34050,0.04200,23.225
2,26,Aigues Mortes,Les Appartement du Mas - RUBIS,https://www.booking.com/hotel/fr/les-apparteme...,43.581269,4.228718,9.6,13.0,"Situé à Aigues-Mortes, l’hébergement Les Appar...",43.566152,4.191540,68.4,6.65950,3.6815,4.32275,0.00000,12.550
3,20,Cassis,La douceur de Cassis,https://www.booking.com/hotel/fr/la-douceur-de...,43.215009,5.530956,9.4,95.0,L’hébergement La douceur de Cassis se trouve à...,43.214036,5.539632,69.4,9.34600,7.4960,3.34050,0.04200,23.225
4,32,Montauban,Le Passage de la Comédie - Climatisation & WiF...,https://www.booking.com/hotel/fr/le-passage-de...,44.018806,1.355119,9.3,35.0,L’hébergement Le Passage de la Comédie - Clima...,44.054015,1.493887,68.0,5.03100,3.5610,2.07700,0.00000,31.225
5,20,Cassis,LOU CIGALOU,https://www.booking.com/hotel/fr/lou-cigalou-c...,43.214926,5.532989,9.3,34.0,"Situé à Cassis, l’hébergement LOU CIGALOU offr...",43.214036,5.539632,69.4,9.34600,7.4960,3.34050,0.04200,23.225
6,20,Cassis,ECHO DES FLOTS vue mer,https://www.booking.com/hotel/fr/echo-des-flot...,43.211916,5.530918,9.2,26.0,"Bénéficiant d’un emplacement en bord de mer, l...",43.214036,5.539632,69.4,9.34600,7.4960,3.34050,0.04200,23.225
7,26,Aigues Mortes,La Villa Mazarin,https://www.booking.com/hotel/fr/la-villa-maza...,43.564987,4.191752,9.1,1183.0,La Villa Mazarin vous accueille dans un bâtime...,43.566152,4.191540,68.4,6.65950,3.6815,4.32275,0.00000,12.550
8,20,Cassis,Hôtel Particulier Cassis - HPC Suites Cassis C...,https://www.booking.com/hotel/fr/hpc-suites.fr...,43.216440,5.541746,9.1,787.0,Doté d’une piscine extérieure commune avec ter...,43.214036,5.539632,69.4,9.34600,7.4960,3.34050,0.04200,23.225
9,32,Montauban,Dali Hôtel Montauban,https://www.booking.com/hotel/fr/dali-montauba...,44.020996,1.349240,8.9,2144.0,"Situé à Montauban, l’établissement Dali Hôtel ...",44.054015,1.493887,68.0,5.03100,3.5610,2.07700,0.00000,31.225


In [55]:
# Map of the top 5 destinations in France for the next 5 days
fig = px.scatter_map(
    df_top5_city,
    lat="city_lat",
    lon="city_lon",
    color="hci",
    size="hci",
    hover_name="city",
    zoom=6,
    title="Top 5 destinations in France for the next 5 days"
)
fig.update_layout(title_x=0.5)
fig.show()

In [56]:
df_top20_hotel.head()

,city_id,city,hotel_name,hotel_url,hotel_latitude,hotel_longitude,hotel_rating,hotel_reviews,hotel_description,city_lat,city_lon,hci,temp,temp_feels_like,wind_speed,rain,cloud
0,27,Saintes Maries de la mer,Résidence L'Oiseau des Mers,https://www.booking.com/hotel/fr/residence-l-o...,43.454050,4.427996,9.9,73.0,L’établissement Résidence L'Oiseau des Mers se...,43.451592,4.427720,67.6,7.22325,3.6560,6.71400,0.003,18.775
1,20,Cassis,Le Bel Ecrin par Dodo-a-Cassis,https://www.booking.com/hotel/fr/le-bon-crin.f...,43.214187,5.534756,9.7,47.0,"Situé à Cassis, l’hébergement Le Bel Ecrin par...",43.214036,5.539632,69.4,9.34600,7.4960,3.34050,0.042,23.225
2,26,Aigues Mortes,Les Appartement du Mas - RUBIS,https://www.booking.com/hotel/fr/les-apparteme...,43.581269,4.228718,9.6,13.0,"Situé à Aigues-Mortes, l’hébergement Les Appar...",43.566152,4.191540,68.4,6.65950,3.6815,4.32275,0.000,12.550
3,20,Cassis,La douceur de Cassis,https://www.booking.com/hotel/fr/la-douceur-de...,43.215009,5.530956,9.4,95.0,L’hébergement La douceur de Cassis se trouve à...,43.214036,5.539632,69.4,9.34600,7.4960,3.34050,0.042,23.225
4,32,Montauban,Le Passage de la Comédie - Climatisation & WiF...,https://www.booking.com/hotel/fr/le-passage-de...,44.018806,1.355119,9.3,35.0,L’hébergement Le Passage de la Comédie - Clima...,44.054015,1.493887,68.0,5.03100,3.5610,2.07700,0.000,31.225


In [60]:
# Map of the top 20 hotels located in the top 5 destinations
fig = px.scatter_map(
    df_top20_hotel,
    lat="hotel_latitude",
    lon="hotel_longitude",
    color="hotel_rating",
    size="hotel_rating",
    hover_name="city",
    title="Top 20 hotels for the 5 best destinations in France for the next 5 days"
)
fig.update_layout(title_x=0.5)
fig.show()

In [62]:
pio.write_html(fig, "outputs/map_top5_destinations.html")
pio.write_html(fig, "outputs/map_top20_hotels.html")
#pio.write_image(fig, "outputs/map_top5_destinations.png")
#pio.write_image(fig, "outputs/map_top20_hotels.png")

Within the 5 next days, the top 5 destinations in France are located in the South East of the country.